In [ ]:
### My libraries (not pip)
try:
    import mathlib as Mathlib   #< c++ version (much faster training). Must be compiled first
    from NeuralNetwork_CPP import Batch, ActivationType
    print("Using C++ compiled network!")
except ImportError:
    import PYTHON_Network.Mathlib as Mathlib
    from PYTHON_Network.ActivationTypes import ActivationType
    from PYTHON_Network.Batch import Batch
    print("Using python network as the C++ compiled libraries are not avaialble!")

import PYTHON_Network.Loss as Loss
import PYTHON_Network.Optimizer as Optimizer
import PYTHON_Network.Accuracy as Accuracy


Using C++ compiled mathlib!


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



In [3]:
import json
import time
start = time.time()

with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

BATCH_SIZE = int(len(X)/20)  #< batch size must be divisible by the number of samples

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")

layer1 = Batch(BATCH_SIZE, 32*32, 32, ActivationType.LEAKY_RELU)
layer2 = Batch(BATCH_SIZE, 32, 2, ActivationType.PASS)

try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    layer1.setWeights(savedModel["layer1"]["weights"])
    layer1.setBiases(savedModel["layer1"]["biases"])
    layer2.setWeights(savedModel["layer2"]["weights"])
    layer2.setBiases(savedModel["layer2"]["biases"])
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")

# softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy_batch()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = layer1.forward(X_batch)                                    #< returns a list of lists of outputs
        layer2Output = layer2.forward(layer1Output)                               #< returns a list of lists of outputs
        # print(layer2Output)
        error = softmaxCrossEntropy.forward(layer2Output, y_batch)                #< returns mean error
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = layer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = layer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(layer2)
        spiralOptimizer.update(layer1)

    if epoch%10 == 0:
        print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")  #< Normal to bounce around at the start
        modelState = {
            "layer1": {
                "weights": layer1.getWeights(),
                "biases": layer1.getBiases()
            },
            "layer2": {
                "weights": layer2.getWeights(),
                "biases": layer2.getBiases()
            }
        }
        with open("trainedModel.json", "w") as f:
            json.dump(modelState, f, indent=4)

end = time.time()
print(f"Training took: {end-start} seconds")


Converting data to hilbert space
Done/1000
Error: 'trainedModel.json' not found. Training from scratch.
Epoch 0: Error: 0.7150544757099841, accuracy: 0.501
Epoch 10: Error: 0.6662485786169094, accuracy: 0.5745454545454546
Epoch 20: Error: 0.6333076437319672, accuracy: 0.6091904761904762
Epoch 30: Error: 0.6058089948540578, accuracy: 0.6309354838709678
Epoch 40: Error: 0.5806686077294129, accuracy: 0.6470975609756098
Epoch 50: Error: 0.5579886642192242, accuracy: 0.6624313725490196
Epoch 60: Error: 0.5496643444824592, accuracy: 0.6747704918032786
Epoch 70: Error: 0.5360397575915717, accuracy: 0.684943661971831
Epoch 80: Error: 0.5274796824625722, accuracy: 0.693358024691358
Epoch 90: Error: 0.50869139269344, accuracy: 0.7017802197802198
Epoch 100: Error: 0.5064605423594865, accuracy: 0.7102970297029703
Epoch 110: Error: 0.4795245362948278, accuracy: 0.7184144144144144
Epoch 120: Error: 0.39143300957904187, accuracy: 0.725694214876033
Epoch 130: Error: 0.38207536902019695, accuracy: 0.73

### The raw Python model takes about 45 minutes to train to 90% and slightly less than 90 minutes to fully train.

### The C++ model takes about 2 minutes to train to 90% accuracy and slightly more than 4 minutes to fully train Thats 21x the speed!

In [ ]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()